In [0]:
%sql
SELECT * FROM gizmobox.bronze.v_orders;

1. Pre-process the JSOn String to fix the Data quality Issues

[Documentation](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace)

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW tv_orders_fixed AS

SELECT value,
regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_values

FROM gizmobox.bronze.v_orders;

2.Transform JSON String to JSON object

Function [schema_of_JSON](https://docs.databricks.com/aws/en/sql/language-manual/functions/schema_of_json)

Function [from_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json)

In [0]:
%sql
SELECT * FROM tv_orders_fixed;
    

In [0]:
%sql
SELECT schema_of_json(fixed_values) FROM tv_orders_fixed limit 1;

In [0]:
%sql
SELECT from_json(fixed_values, 
'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'
) AS json_values

FROM tv_orders_fixed;

3.Write transformed data to the silver schema

In [0]:
%sql
CREATE TABLE gizmobox.silver.orders_json
AS
SELECT from_json(fixed_values, 
'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'
) AS json_values
FROM tv_orders_fixed;

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders_json